# Superhero Intelligence System
**Can six ability scores tell a hero from a villain?**

Sections: 2 Data Preparation · Task 1 Classification · Task 2 Clustering · Task 3 Neural Network

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

SCORES = ["intelligence_score", "strength_score", "speed_score",
          "durability_score", "power_score", "combat_score"]
COLORS = {"hero": "#1F5FBF", "villain": "#D62839", "baseline": "#9E9E9E",
          "tree": "#2E7D32", "knn": "#F9A825", "netA": "#6A1B9A", "netB": "#00838F"}

## Section 2: Data Preparation

### Step 1: Inspect (T1)

In [ ]:
train_raw = pd.read_csv("data/train.csv")
test_raw = pd.read_csv("data/test.csv")

def inspect(df, label):
    print(f"== {label}: shape {df.shape}")
    use = SCORES + ["alignment", "name"]
    num = df[SCORES].apply(pd.to_numeric, errors="coerce")
    t = pd.DataFrame({
        "dtype": df[use].dtypes.astype(str),
        "missing": df[use].isna().sum(),
        "zeros": [(num[c] == 0).sum() if c in SCORES else np.nan for c in use],
    })
    # blanks/non-numeric that become NaN + zeros that become NaN
    t["-> NaN (total)"] = [num[c].isna().sum() + (num[c] == 0).sum() if c in SCORES else np.nan for c in use]
    display(t)

print("Columns:", list(train_raw.columns))
inspect(train_raw, "train.csv"); inspect(test_raw, "test.csv")

### Steps 2-4: duplicates, mark missing, drop weak rows (T2)

In [ ]:
n = {"Loaded": (len(train_raw), len(test_raw))}
cols = ["name", "alignment"] + SCORES
train, test = train_raw[cols].copy(), test_raw[cols].copy()

# Step 2: duplicates
train = train.drop_duplicates("name", keep="first")
test = test.drop_duplicates("name", keep="first")
train = train[~train["name"].isin(test["name"])]
n["After step 2 (duplicates)"] = (len(train), len(test))

# Step 3: blanks and 0 -> NaN
def mark_missing(df):
    df = df.copy()
    df[SCORES] = df[SCORES].apply(pd.to_numeric, errors="coerce").replace(0, np.nan)
    return df
train, test = mark_missing(train), mark_missing(test)

# Step 4: drop rows with 3+ missing scores
train = train[train[SCORES].isna().sum(axis=1) < 3]
test = test[test[SCORES].isna().sum(axis=1) < 3]
n["After step 4 (weak rows)"] = (len(train), len(test))

### Clean Set A (clustering) and Clean Set B (classification / NN)

In [ ]:
set_a = train.reset_index(drop=True)                      # Task 2: all characters, train only

def set_b(df):
    df = df[df["alignment"].isin(["Good", "Bad"])].copy()
    df["target"] = df["alignment"].map({"Good": 0, "Bad": 1})
    return df.reset_index(drop=True)
train_b, test_b = set_b(train), set_b(test)               # Tasks 1 & 3

n["Clean Set A"] = (len(set_a), "not used")
n["Clean Set B"] = (len(train_b), len(test_b))
T2 = pd.DataFrame(n, index=["train.csv", "test.csv"]).T
display(T2)
print("Removed from Set B (Neutral/blank alignment): train", len(train) - len(train_b),
      "| test", len(test) - len(test_b))
print("Alignment values in cleaned train:", train["alignment"].value_counts(dropna=False).to_dict())

## Task 1: Classification (25 marks)
_Baseline, Decision Tree, KNN on Clean Set B. Test once at the end._

## Task 2: Clustering (25 marks)
_K-Means and Agglomerative on Clean Set A. `alignment` only looked at after clustering._

## Task 3: Neural Network (25 marks)
_Network A (6-8-1) and Network B (6-16-8-1). Final ranking of all 5 models._